## נתונים מול מודל: הגרף המלא

בעמוד זה כל חלקי הניתוח נאספים לגרף אחד: נתונים עם שגיאות (‎`errorbar`‎), קו ההתאמה, השאריות וסיכום מספרי (‎`chi2_red`‎).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

g = 9.8
df = pd.read_csv("lab_measurements.csv")
df_clean = df.dropna()
angles = sorted(df_clean["angle_deg"].unique())

x = np.array([np.sin(2*np.radians(a)) for a in angles])
y = np.array([df_clean[df_clean["angle_deg"] == a]["range_measured"].mean() for a in angles])
dy = np.array([
    df_clean[df_clean["angle_deg"] == a]["range_measured"].std(ddof=1) / np.sqrt(len(df_clean[df_clean["angle_deg"] == a]))
    for a in angles
])

def linear_fit(x, y):
    x_bar, y_bar = x.mean(), y.mean()
    a1 = ((x - x_bar) * (y - y_bar)).sum() / ((x - x_bar)**2).sum()
    a0 = y_bar - a1 * x_bar
    return a0, a1

a0, a1 = linear_fit(x, y)
x_line = np.linspace(0, 1, 100)
y_line = a0 + a1*x_line
resid = y - (a0 + a1*x)

chi2 = np.sum((resid/dy)**2)
chi2_red = chi2 / (len(x) - 2)

### הגרף המלא: נתונים + התאמה + שאריות

כמו בסעיף 8.6, ‎`plt.subplots(2, 1, ...)`‎ עם ‎`sharex=True`‎ יוצר שני חלונות זה מעל זה, עם ציר x משותף. שלושה פרטים חדשים בקוד: ‎`(ax1, ax2)`‎ פורק את שני הצירים לשני משתנים, במקום ‎`axs[0]`‎ ו-‎`axs[1]`‎; ‎`height_ratios=[2, 1]`‎ נותן לחלון העליון גובה כפול מזה של התחתון; ו-‎`plt.tight_layout()`‎ מרווח בין החלונות כך שהתוויות אינן חופפות.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6, 6), height_ratios=[2, 1], sharex=True)

ax1.errorbar(x, y, yerr=dy, fmt="o", capsize=3,
             label="Measurement (mean per angle)")
ax1.plot(x_line, y_line, "-", label=f"Fit: R = {a0:.2f} + {a1:.2f}·sin(2θ)")
ax1.set_ylabel("Mean range [m]")
ax1.set_title(f"chi²_ν = {chi2_red:.2f}")
ax1.legend()

ax2.axhline(0, color="gray", linewidth=1)
ax2.errorbar(x, resid, yerr=dy, fmt="o", color="tab:red")
ax2.set_xlabel("sin(2θ)")
ax2.set_ylabel("Residual [m]")

plt.tight_layout()
plt.show()

### נסו בעצמכם

השארית המנורמלת, ‎`resid / dy`‎, מראה כמה שגיאות מדידה כל נקודה רחוקה מהמודל. חשבו אותה במערך ‎`normalized_resid`‎, ובדקו שסכום הריבועים שלה שווה ל-‎`chi2`‎. אחר כך ציירו גרף של ‎`normalized_resid`‎ מול ‎`x`‎, עם קו אופקי ב-0 וקווים מקווקווים ב-$1$ וב-$-1$. כמה נקודות נמצאות מחוץ לתחום שבין שני הקווים המקווקווים?

In [ ]:
# normalized_resid = ...
# print(...)
# fig, ax = plt.subplots()
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
normalized_resid = resid / dy
print(np.sum(normalized_resid**2), chi2)

fig, ax = plt.subplots()
ax.axhline(0, color="gray", linewidth=1)
ax.axhline(1, color="gray", linestyle="--")
ax.axhline(-1, color="gray", linestyle="--")
ax.plot(x, normalized_resid, "o", color="tab:red")
ax.set_xlabel("sin(2θ)")
ax.set_ylabel("Residual / dy")
plt.show()
```
שני המספרים שמודפסים זהים: χ² הוא בדיוק סכום הריבועים של השאריות המנורמלות. נקודה אחת, זווית 30 מעלות, נמצאת מחוץ לתחום שבין $-1$ ל-$1$.
`````

### בדקו את עצמכם

In [ ]:
import json
from jupyterquiz import display_quiz

example = \
'''
[
    {
        "question": "מדוע שני חלונות הגרף המלא חולקים את אותו ציר x?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "כדי שכל שארית תופיע בדיוק מתחת לנקודה שלה בחלון העליון",
                "correct": true,
                "feedback": "נכון - כך קל לראות לאיזו מדידה שייכת כל שארית."
            },
            {
                "answer": "כדי ששני החלונות יקבלו את אותו ציר y",
                "correct": false,
                "feedback": "לא - ציר x משותף אינו משנה את ציר y; לכל חלון ציר y משלו."
            },
            {
                "answer": "כי פסי השגיאה בחלון העליון דורשים זאת",
                "correct": false,
                "feedback": "לא - פסי שגיאה אפשר לצייר גם בלי ציר משותף."
            },
            {
                "answer": "כדי לחסוך בזיכרון",
                "correct": false,
                "feedback": "לא - הסיבה היא קריאות הגרף, לא זיכרון."
            }
        ]
    },
    {
        "question": "בכותרת הגרף χ² המצומצם הוא כ-1.3. מה המשמעות?",
        "type": "multiple_choice",
        "answers": [
            {
                "answer": "השאריות בגודל שמתאים לשגיאות המדידה, וההתאמה טובה",
                "correct": true,
                "feedback": "נכון - ערך קרוב ל-1 הוא בדיוק מה שמצפים מהתאמה טובה."
            },
            {
                "answer": "המודל שגוי לחלוטין",
                "correct": false,
                "feedback": "לא - מודל שגוי היה נותן χ² מצומצם גדול בהרבה מ-1."
            },
            {
                "answer": "שגיאות המדידה הוערכו ביתר",
                "correct": false,
                "feedback": "לא - הערכת יתר של שגיאות המדידה נותנת χ² מצומצם קטן בהרבה מ-1."
            },
            {
                "answer": "הקו עובר בדיוק דרך כל הנקודות",
                "correct": false,
                "feedback": "לא - אז כל השאריות היו 0, וגם χ² היה 0."
            }
        ]
    }
]
'''
myquiz = json.loads(example)
display_quiz(myquiz)